In [ ]:
'''
Train a Transformer encoder model to generate MIDI:
    1. pre-train
    (2. SFT)
    (3. RL)
'''
from pathlib import Path
from random import shuffle

from miditok import REMI, TokenizerConfig
from miditok.utils import split_files_for_training

from config import ModelParams, HyperParams

PREPROCESS = False # set to True if you haven't run PREPROCESS yet
TRAIN = True # set to False if you want to skip training

TOKENIZER_PARAMS = {
    "special_tokens": ["PAD_None", "MASK_None"],
    "pitch_range": (22, 107),
    "num_velocities": 32,
    "beat_res": {(0, 8): 4}, # 16 Position tokens within a bar
    "use_pitchdrum_tokens": False,
    "use_chords": False,
    "chord_tokens_with_root_note": False,
}
config = TokenizerConfig(**TOKENIZER_PARAMS)
tokenizer = REMI(config)

if PREPROCESS:
    midi_paths = list(Path(HyperParams.DATA_PATH).rglob("*.mid"))
    shuffle(midi_paths)
    midi_paths = [p.resolve() for p in midi_paths if p.is_file()]
    total_num_files = len(midi_paths)

    tokenizer.save(Path("models", HyperParams.name + "_tokenizer.json"))

    num_files_valid = round(total_num_files * 0.05)
    num_files_test = round(total_num_files * 0.05)
    midi_paths_val = midi_paths[:num_files_valid]
    midi_paths_test = midi_paths[num_files_valid:num_files_valid + num_files_test]
    midi_paths_train = midi_paths[num_files_valid + num_files_test:]
    
    # Chunk MIDIs and perform data augmentation on each subset independently
    for files_paths, subset_name in (
        (midi_paths_train, "train"), (midi_paths_val, "val"), (midi_paths_test, "test")
    ):
        subset_chunks_dir = Path(f"pre-train_dataset/{subset_name}")
        split_files_for_training(
            files_paths=files_paths,
            tokenizer=tokenizer,
            save_dir=subset_chunks_dir,
            max_seq_len=1024,
            num_overlap_bars=2,
        )
else:
    tokenizer._load_from_json(Path("models", HyperParams.name + "_tokenizer.json"))

print(tokenizer.vocab)
print(tokenizer.vocab_size)

In [ ]:
from help import MIDIDataset, DataCollator
from torch.utils.data import DataLoader

midi_paths_train = list(Path("pre-train_dataset", "train").rglob("*.mid"))
midi_paths_val = list(Path("pre-train_dataset", "val").rglob("*.mid"))

train_dataset = MIDIDataset(midi_paths_train, tokenizer)
val_dataset = MIDIDataset(midi_paths_val, tokenizer)

# Extend sequences with PAD tokens (PAD tokens are treated as normal tokens during training)
collator = DataCollator(pad_token_id=tokenizer.vocab['PAD_None'])

train_dataloader = DataLoader(dataset=train_dataset, batch_size=HyperParams.batch_size, collate_fn=collator, shuffle=True)
val_dataloader = DataLoader(dataset=val_dataset, batch_size=HyperParams.batch_size, collate_fn=collator, shuffle=True)

print(len(train_dataset), "train samples")
print(len(val_dataset), "validation samples")

In [ ]:
from help import Trainer
from tqdm.notebook import tqdm

trainer = Trainer(ModelParams, HyperParams, tokenizer)

print(sum(p.numel() for p in trainer.mask_predictor.parameters()), "model parameters\n")
print("Model architecture:")
print(trainer.mask_predictor)

if HyperParams.retrain:
    # load model and its optimizer from a checkpoint (fine_tune=True -> load only model)
    loss = trainer.load("models/checkpoints/pre-training_792000_checkpoint.pth", fine_tune=True)

In [ ]:
if TRAIN:
    from torch.utils.tensorboard import SummaryWriter
    from itertools import cycle
    
    # Setup
    trainer.phase = 'pre-training' # 'pre-training' or 'SFT'
    model_dir = 'models/checkpoints/' + trainer.phase + '_'

    writer = SummaryWriter()

    global_step = 0

    # Optional: wrap dataloader with cycle to avoid worrying about epoch ends
    train_iterator = cycle(train_dataloader)

    pbar = tqdm(total=HyperParams.num_training_steps, desc="Training", position=0)

    best_val = float('inf')
    train_loss = 0

    trainer.mask_predictor.train()
    while global_step < HyperParams.num_training_steps:
        train_batch = next(train_iterator)

        train_loss += trainer.train_step(train_batch)

        # Logging
        global_step += 1
        if global_step % HyperParams.logging_interval == 0:
            val_loss = 0

            trainer.mask_predictor.eval()
            for val_batch in tqdm(val_dataloader, desc="Validating", leave=False):
                val_loss += trainer.val_step(val_batch)
            trainer.mask_predictor.train()

            writer.add_scalar('Loss/val', val_loss / len(val_dataloader), global_step)       
            writer.add_scalar('Loss/train', train_loss / HyperParams.logging_interval, global_step)
            writer.add_scalar('LR', trainer.lr_scheduler.get_last_lr()[0], global_step)

            if val_loss < best_val:
                trainer.save('models/best_val', val_loss / len(val_dataloader))
                best_val = val_loss

            trainer.save(model_dir + str(global_step), train_loss / HyperParams.logging_interval)
            train_loss = 0

            trainer.mask_predictor.train()
        pbar.update(1)

    pbar.close()
    writer.close()